# Etapa 1: curadoria

Entrada: `data/raw/adult.csv`, baixado do Kaggle (*Adult Census Income*, `uciml/adult-census-income`). É a base *Adult* do repositório UCI, extraída por Barry Becker do censo dos Estados Unidos de 1994: uma linha por pessoa, com dados demográficos e de trabalho, e o alvo diz se a pessoa ganha mais de 50 mil dólares por ano. O arquivo do Kaggle tem o mesmo conteúdo do `adult.data` da UCI, com cabeçalho e as linhas em outra ordem.

Saída: `data/processed/renda.csv`, sem linhas duplicadas, sem valores ausentes, sem as colunas `fnlwgt` e `education` e com o alvo em 0/1.

## 0. Configuração

In [1]:
import sys
import hashlib
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config

sha256 = hashlib.sha256(config.RAW_CSV.read_bytes()).hexdigest().upper()
print("SHA-256 confere:", sha256 == config.RAW_SHA256)

bruto = pd.read_csv(config.RAW_CSV)
print(bruto.shape)
bruto.head()

SHA-256 confere: True
(32561, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


## 1. Primeira olhada

In [2]:
texto = bruto.select_dtypes(exclude="number").columns.tolist()
numero = bruto.select_dtypes(include="number").columns.tolist()
print("numéricas:", numero)
print("texto:", texto)
print("valores com espaço no começo ou no fim:",
      int(sum((bruto[c] != bruto[c].str.strip()).sum() for c in texto)))
print("NaN reconhecidos pelo pandas:", int(bruto.isna().sum().sum()))
print()
print(bruto[config.TARGET].value_counts())
print((bruto[config.TARGET].value_counts(normalize=True) * 100).round(1))

numéricas: ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
texto: ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country', 'income']
valores com espaço no começo ou no fim: 0
NaN reconhecidos pelo pandas: 0

income
<=50K    24720
>50K      7841
Name: count, dtype: int64
income
<=50K    75.9
>50K     24.1
Name: proportion, dtype: float64


São 32.561 pessoas e 15 colunas: 6 numéricas e 9 de texto (contando o alvo). Os textos já vêm sem espaços sobrando, e o pandas não encontra nenhum valor faltante. Isso não quer dizer que a base esteja completa: os ausentes estão marcados com `"?"`, que o pandas lê como um texto qualquer (seção 3).

O alvo é desbalanceado: 75,9% ganham até 50 mil dólares e 24,1% ganham mais, cerca de 3 para 1.

## 2. Linhas duplicadas

In [3]:
duplicadas = bruto[bruto.duplicated(keep=False)].sort_values(bruto.columns.tolist())
print("linhas repetidas (cópias a mais):", int(bruto.duplicated().sum()))
print("linhas envolvidas, contando a original:", len(duplicadas))
duplicadas.head(4)

linhas repetidas (cópias a mais): 24
linhas envolvidas, contando a original: 47


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
19622,19,Private,97261,HS-grad,9,Never-married,Farming-fishing,Not-in-family,White,Male,0,0,40,United-States,<=50K
20507,19,Private,97261,HS-grad,9,Never-married,Farming-fishing,Not-in-family,White,Male,0,0,40,United-States,<=50K
10307,19,Private,138153,Some-college,10,Never-married,Adm-clerical,Own-child,White,Female,0,0,10,United-States,<=50K
22783,19,Private,138153,Some-college,10,Never-married,Adm-clerical,Own-child,White,Female,0,0,10,United-States,<=50K


Há 24 cópias a mais, 0,07% da base. Num censo, duas pessoas diferentes podem ter as mesmas respostas, mas aqui as cópias coincidem até no `fnlwgt`, um peso com seis dígitos, o que torna a repetição por acaso improvável. Além disso, uma linha repetida pode cair no treino e no teste ao mesmo tempo, e o modelo seria avaliado numa pessoa que já viu. As cópias são removidas, mantendo a primeira ocorrência.

In [4]:
base = bruto.drop_duplicates().reset_index(drop=True)
print(base.shape)

(32537, 15)


## 3. Valores ausentes marcados com "?"

In [5]:
ausente = base == "?"
print('valores "?" por coluna:')
print(ausente.sum()[ausente.sum() > 0])
print()
print('linhas com algum "?":', int(ausente.any(axis=1).sum()),
      f"({ausente.any(axis=1).mean() * 100:.1f}%)")
print()
print("classe de trabalho × ocupação ausentes:")
print(pd.crosstab(ausente["workclass"], ausente["occupation"],
                  rownames=["workclass = ?"], colnames=["occupation = ?"]))
print()
so_ocupacao = base[ausente["occupation"] & ~ausente["workclass"]]
print('ocupação "?" com classe de trabalho preenchida:', so_ocupacao["workclass"].value_counts().to_dict())

valores "?" por coluna:
workclass         1836
occupation        1843
native.country     582
dtype: int64

linhas com algum "?": 2398 (7.4%)

classe de trabalho × ocupação ausentes:
occupation = ?  False  True 
workclass = ?               
False           30694      7
True                0   1836

ocupação "?" com classe de trabalho preenchida: {'Never-worked': 7}


Os `"?"` aparecem em três colunas: classe de trabalho, ocupação e país de origem. Classe de trabalho e ocupação faltam quase sempre juntas. As 7 linhas em que só a ocupação falta são de pessoas que nunca trabalharam, para quem a ocupação de fato não existe.

Antes de remover, vale ver se as linhas com `"?"` são parecidas com as demais.

In [6]:
com = ausente.any(axis=1)
grupo = com.map({True: 'com "?"', False: 'sem "?"'})
taxa = base.groupby(grupo)[config.TARGET].apply(lambda s: round((s == config.POSITIVO).mean() * 100, 1))
print("% que ganha >50K:")
print(taxa)

% que ganha >50K:
com "?"    13.9
sem "?"    24.9
Name: income, dtype: float64


Os ausentes não estão espalhados ao acaso: entre as pessoas com `"?"`, só 13,9% ganham mais de 50 mil, contra 24,9% nas demais. O perfil sugere pessoas fora do mercado de trabalho ou com vínculo informal. Remover essas linhas deixa a base um pouco mais "rica" e tira do modelo esse perfil; a alternativa seria tratar `"?"` como uma categoria própria.

Optamos por remover: são 7,4% das linhas, sobram mais de 30 mil pessoas, e é o tratamento mais usado na literatura sobre esta base, o que facilita comparar resultados. O viés citado acima fica registrado como limitação.

In [7]:
base = base[~com].reset_index(drop=True)
print(base.shape)
print('"?" restantes:', int((base == "?").sum().sum()))

(30139, 15)
"?" restantes: 0


## 4. Colunas redundantes ou sem significado para o modelo

**Escolaridade.** `education` (texto) e `education.num` (número) parecem dizer a mesma coisa. Se cada nível de escolaridade corresponder a um único número, uma das duas sobra.

In [8]:
print("números distintos por nível:", base.groupby("education")["education.num"].nunique().max())
niveis = base.groupby("education")["education.num"].agg(numero="first", pessoas="size")
niveis.sort_values("numero")

números distintos por nível: 1


,numero,pessoas
education,,
Preschool,1,44
1st-4th,2,149
5th-6th,3,287
7th-8th,4,556
9th,5,455
10th,6,820
11th,7,1048
12th,8,377
HS-grad,9,9834


Cada nível tem exatamente um número, e a ordem dos números segue a ordem dos níveis: 1 é pré-escola, 9 é ensino médio completo (*HS-grad*), 13 é graduação (*Bachelors*) e 16 é doutorado. Fica `education.num`: escolaridade tem ordem natural, e o número a preserva sem precisar de 16 colunas de one-hot. `education` sai.

**Peso amostral.** `fnlwgt` (*final weight*) é o peso que o órgão do censo atribui a cada pessoa entrevistada: quantas pessoas da população ela representa, calculado a partir de idade, sexo, raça e estado. Ele descreve como a amostra foi feita, não a pessoa.

In [9]:
alvo01 = (base[config.TARGET] == config.POSITIVO).astype(int)
print("correlação entre fnlwgt e o alvo:", round(base["fnlwgt"].corr(alvo01), 3))
print(base["fnlwgt"].describe().round(0))

correlação entre fnlwgt e o alvo: -0.009
count      30139.0
mean      189795.0
std       105659.0
min        13769.0
25%       117628.0
50%       178417.0
75%       237604.0
max      1484705.0
Name: fnlwgt, dtype: float64


A correlação com o alvo é praticamente zero. Sem significado para o problema e sem relação com a renda, `fnlwgt` sai.

In [10]:
base = base.drop(columns=config.REMOVIDAS)
print(list(base.columns))

['age', 'workclass', 'education.num', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'capital.gain', 'capital.loss', 'hours.per.week', 'native.country', 'income']


**Perfis repetidos.** Sem o `fnlwgt`, pessoas diferentes que responderam o mesmo ao censo ficam com linhas idênticas. Elas não são cópias: as 24 cópias verdadeiras (iguais até no peso) já saíram na seção 2. Vale medir quantas são e se o mesmo perfil aparece com rendas diferentes, olhando só os atributos que os modelos vão usar.

In [11]:
perfil = base[config.FEATURES]
repetem = perfil.duplicated()
print("linhas cujo perfil já apareceu em outra linha:", int(repetem.sum()), f"({repetem.mean() * 100:.1f}%)")

rendas = base.groupby(config.FEATURES)[config.TARGET].nunique()
conflito = rendas[rendas > 1]
linhas_conflito = base.set_index(config.FEATURES).index.isin(conflito.index)
print("perfis que aparecem com as duas rendas:", len(conflito))
print("linhas nesses perfis:", int(linhas_conflito.sum()), f"({linhas_conflito.mean() * 100:.1f}%)")

# Num perfil com as duas rendas, o melhor que um modelo pode fazer é responder a renda
# mais comum do perfil; as pessoas da outra renda serão erros de qualquer jeito.
contagem = base.assign(rico=alvo01).groupby(config.FEATURES)["rico"].agg(ricos="sum", pessoas="size")
contagem["outros"] = contagem["pessoas"] - contagem["ricos"]
erro_minimo = int(contagem[["ricos", "outros"]].min(axis=1).sum())
print("erros inevitáveis:", erro_minimo, f"({erro_minimo / len(base) * 100:.1f}% da base)")
print(f"acurácia máxima possível: {(1 - erro_minimo / len(base)) * 100:.1f}%")

linhas cujo perfil já apareceu em outra linha: 5057 (16.8%)
perfis que aparecem com as duas rendas: 673
linhas nesses perfis: 2404 (8.0%)
erros inevitáveis: 867 (2.9% da base)
acurácia máxima possível: 97.1%


Os perfis repetidos são os mais comuns (jovens solteiros, 40 horas por semana, sem ganho de capital), e removê-los apagaria pessoas reais e distorceria a base justamente onde ela é mais densa. Eles ficam. Na divisão em treino e teste, uma pessoa pode ter um "gêmeo" do outro lado; isso não é vazamento, porque são pessoas distintas, e um modelo em uso também encontraria perfis já vistos.

Os 673 perfis que aparecem com as duas rendas mostram um limite: para essas pessoas, os atributos disponíveis não bastam para separar as classes. O melhor que um modelo pode fazer é responder a renda mais comum de cada perfil, e as demais pessoas viram erros de qualquer jeito: são 867 erros inevitáveis, 2,9% da base. Nenhum classificador passa de 97,1% de acurácia com estes atributos.

## 5. Alvo em 0/1

O alvo vira 1 para quem ganha mais de 50 mil dólares (a classe de interesse) e 0 para os demais. Os atributos de texto continuam como texto: a conversão em números (one-hot) é feita dentro do pipeline de cada modelo, ajustada só no treino.

Raça e sexo continuam na base, mas não entram nos modelos (`config.SENSIVEIS`): servem para a análise exploratória e para conferir, depois da previsão, se o modelo erra mais com algum grupo.

In [12]:
base[config.TARGET] = (base[config.TARGET] == config.POSITIVO).astype(int)
base = base[config.NUMERICAS + config.CATEGORICAS + config.SENSIVEIS + [config.TARGET]]
base.head()

,age,education.num,capital.gain,capital.loss,hours.per.week,workclass,marital.status,occupation,relationship,native.country,race,sex,income
0,82,9,0,4356,18,Private,Widowed,Exec-managerial,Not-in-family,United-States,White,Female,0
1,54,4,0,3900,40,Private,Divorced,Machine-op-inspct,Unmarried,United-States,White,Female,0
2,41,10,0,3900,40,Private,Separated,Prof-specialty,Own-child,United-States,White,Female,0
3,34,9,0,3770,45,Private,Divorced,Other-service,Unmarried,United-States,White,Female,0
4,38,6,0,3770,40,Private,Separated,Adm-clerical,Unmarried,United-States,White,Male,0


## 6. Base final

In [13]:
assert len(base) == len(bruto) - 24 - int(com.sum())
assert not (base == "?").any().any()
assert not base.isna().any().any()
assert base[config.TARGET].isin([0, 1]).all()
assert not set(config.REMOVIDAS) & set(base.columns)

config.PROCESSED_CSV.parent.mkdir(parents=True, exist_ok=True)
base.to_csv(config.PROCESSED_CSV, index=False)
print("gravado:", config.PROCESSED_CSV.name, base.shape)
print("ganham >50K:", int(base[config.TARGET].sum()), f"({base[config.TARGET].mean() * 100:.1f}%)")
print("categorias por atributo:", {c: base[c].nunique() for c in config.CATEGORICAS + config.SENSIVEIS})

gravado: renda.csv (30139, 13)
ganham >50K: 7506 (24.9%)
categorias por atributo: {'workclass': 7, 'marital.status': 7, 'occupation': 14, 'relationship': 6, 'native.country': 41, 'race': 5, 'sex': 2}


**Resultado.** 30.139 pessoas e 13 colunas: 5 numéricas e 5 categóricas usadas pelos modelos, 2 sensíveis (raça e sexo, só para análise) e o alvo. Saíram 24 linhas duplicadas, 2.398 linhas com valores ausentes (`"?"`, 7,4% da base) e as colunas `fnlwgt` (peso amostral, correlação de −0,009 com o alvo) e `education` (repete `education.num`). O alvo tem 7.506 pessoas que ganham mais de 50 mil dólares (24,9%), cerca de 3 para 1.

Duas observações para as próximas etapas: os ausentes removidos tinham renda mais baixa que a média (13,9% contra 24,9% de >50K), e 16,8% das linhas repetem o perfil de outra pessoa, com 867 erros inevitáveis (acurácia máxima de 97,1%).